# Explore shufinskiy `nbastats` Data

Goal: inspect one local shufinskiy season before designing the M1 schema.

The key question is whether this file is player-game box scores or raw play-by-play events.

## Setup

Use the already-downloaded CSV so this notebook is deterministic and does not hit `stats.nba.com`.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

csv_path = Path("nbastats_2024.csv")
if not csv_path.exists():
    csv_path = Path("notebooks/nbastats_2024.csv")

csv_path.exists(), csv_path

(True, PosixPath('nbastats_2024.csv'))

In [2]:
# Keep IDs as strings and keep blanks as blanks so participant slots are easy to inspect.
pbp = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
pbp.shape

(574360, 34)

## Raw Columns And Sample Rows

In [3]:
pbp.columns.tolist()

['GAME_ID',
 'EVENTNUM',
 'EVENTMSGTYPE',
 'EVENTMSGACTIONTYPE',
 'PERIOD',
 'WCTIMESTRING',
 'PCTIMESTRING',
 'HOMEDESCRIPTION',
 'NEUTRALDESCRIPTION',
 'VISITORDESCRIPTION',
 'SCORE',
 'SCOREMARGIN',
 'PERSON1TYPE',
 'PLAYER1_ID',
 'PLAYER1_NAME',
 'PLAYER1_TEAM_ID',
 'PLAYER1_TEAM_CITY',
 'PLAYER1_TEAM_NICKNAME',
 'PLAYER1_TEAM_ABBREVIATION',
 'PERSON2TYPE',
 'PLAYER2_ID',
 'PLAYER2_NAME',
 'PLAYER2_TEAM_ID',
 'PLAYER2_TEAM_CITY',
 'PLAYER2_TEAM_NICKNAME',
 'PLAYER2_TEAM_ABBREVIATION',
 'PERSON3TYPE',
 'PLAYER3_ID',
 'PLAYER3_NAME',
 'PLAYER3_TEAM_ID',
 'PLAYER3_TEAM_CITY',
 'PLAYER3_TEAM_NICKNAME',
 'PLAYER3_TEAM_ABBREVIATION',
 'VIDEO_AVAILABLE_FLAG']

In [4]:
pbp.head(10)

,GAME_ID,EVENTNUM,EVENTMSGTYPE,EVENTMSGACTIONTYPE,PERIOD,WCTIMESTRING,PCTIMESTRING,HOMEDESCRIPTION,NEUTRALDESCRIPTION,VISITORDESCRIPTION,SCORE,SCOREMARGIN,PERSON1TYPE,PLAYER1_ID,PLAYER1_NAME,PLAYER1_TEAM_ID,PLAYER1_TEAM_CITY,PLAYER1_TEAM_NICKNAME,PLAYER1_TEAM_ABBREVIATION,PERSON2TYPE,PLAYER2_ID,PLAYER2_NAME,PLAYER2_TEAM_ID,PLAYER2_TEAM_CITY,PLAYER2_TEAM_NICKNAME,PLAYER2_TEAM_ABBREVIATION,PERSON3TYPE,PLAYER3_ID,PLAYER3_NAME,PLAYER3_TEAM_ID,PLAYER3_TEAM_CITY,PLAYER3_TEAM_NICKNAME,PLAYER3_TEAM_ABBREVIATION,VIDEO_AVAILABLE_FLAG
0,22400001,2,12,0,1,7:11 PM,12:00,,Start of 1st Period (7:11 PM EST),,,,0,0,,,,,,0,0,,,,,,0,0,,,,,,0
1,22400001,4,10,0,1,7:11 PM,12:00,Jump Ball Horford vs. Capela: Tip to Wallace,,,,,4,201143,Al Horford,1610612738,Boston,Celtics,BOS,5,203991,Clint Capela,1610612737,Atlanta,Hawks,ATL,5,1630811,Keaton Wallace,1610612737,Atlanta,Hawks,ATL,1
2,22400001,7,2,1,1,7:11 PM,11:43,Tatum BLOCK (1 BLK),,MISS Risacher 27' 3PT Jump Shot,,,5,1642258,Zaccharie Risacher,1610612737,Atlanta,Hawks,ATL,0,0,,,,,,4,1628369,Jayson Tatum,1610612738,Boston,Celtics,BOS,1
3,22400001,9,4,0,1,7:11 PM,11:42,,,Risacher REBOUND (Off:1 Def:0),,,5,1642258,Zaccharie Risacher,1610612737,Atlanta,Hawks,ATL,0,0,,,,,,0,0,,,,,,1
4,22400001,10,2,102,1,7:11 PM,11:38,,,MISS Johnson 14' Driving Floating Bank Jump Shot,,,5,1630552,Jalen Johnson,1610612737,Atlanta,Hawks,ATL,0,0,,,,,,0,0,,,,,,1
5,22400001,11,4,0,1,7:11 PM,11:37,Horford REBOUND (Off:0 Def:1),,,,,4,201143,Al Horford,1610612738,Boston,Celtics,BOS,0,0,,,,,,0,0,,,,,,1
6,22400001,12,2,1,1,7:11 PM,11:24,MISS White 28' 3PT Jump Shot,,,,,4,1628401,Derrick White,1610612738,Boston,Celtics,BOS,0,0,,,,,,0,0,,,,,,1
7,22400001,13,4,0,1,7:11 PM,11:22,,,Johnson REBOUND (Off:0 Def:1),,,5,1630552,Jalen Johnson,1610612737,Atlanta,Hawks,ATL,0,0,,,,,,0,0,,,,,,1
8,22400001,14,5,4,1,7:11 PM,11:17,,,Johnson Traveling Turnover (P1.T1),,,5,1630552,Jalen Johnson,1610612737,Atlanta,Hawks,ATL,0,0,,,,,,1,0,,,,,,1
9,22400001,15,5,1,1,7:12 PM,10:57,Brown Bad Pass Turnover (P1.T1),,Daniels STEAL (1 STL),,,4,1627759,Jaylen Brown,1610612738,Boston,Celtics,BOS,5,1630700,Dyson Daniels,1610612737,Atlanta,Hawks,ATL,0,0,,,,,,1


## Is This Player-Game Data?

A player-game dataset should have one row per player per game and direct stat columns like `PTS`, `REB`, `AST`, and `MIN`.

This file has one row per event.

In [5]:
summary = {
    "rows": len(pbp),
    "columns": pbp.shape[1],
    "unique_games": pbp["GAME_ID"].nunique(),
    "rows_per_game_avg": len(pbp) / pbp["GAME_ID"].nunique(),
}
summary

{'rows': 574360,
 'columns': 34,
 'unique_games': 1230,
 'rows_per_game_avg': 466.9593495934959}

In [6]:
expected_player_log_cols = ["PTS", "REB", "AST", "MIN", "GAME_DATE"]
missing_player_log_cols = [col for col in expected_player_log_cols if col not in pbp.columns]
missing_player_log_cols

['PTS', 'REB', 'AST', 'MIN', 'GAME_DATE']

Conclusion: this is not the direct source for `player_game_logs`. Use `nba_api` for box scores/player game logs. Use shufinskiy for raw play-by-play.

## Game Identity

`GAME_ID` identifies the game. `EVENTNUM` identifies the event within that game. This pair is a candidate natural key, but the raw file should still be checked for duplicates before enforcing a unique constraint.

In [7]:
pbp[["GAME_ID", "EVENTNUM", "EVENTMSGTYPE", "PERIOD", "PCTIMESTRING", "HOMEDESCRIPTION", "VISITORDESCRIPTION", "NEUTRALDESCRIPTION"]].head(20)

,GAME_ID,EVENTNUM,EVENTMSGTYPE,PERIOD,PCTIMESTRING,HOMEDESCRIPTION,VISITORDESCRIPTION,NEUTRALDESCRIPTION
0,22400001,2,12,1,12:00,,,Start of 1st Period (7:11 PM EST)
1,22400001,4,10,1,12:00,Jump Ball Horford vs. Capela: Tip to Wallace,,
2,22400001,7,2,1,11:43,Tatum BLOCK (1 BLK),MISS Risacher 27' 3PT Jump Shot,
3,22400001,9,4,1,11:42,,Risacher REBOUND (Off:1 Def:0),
4,22400001,10,2,1,11:38,,MISS Johnson 14' Driving Floating Bank Jump Shot,
5,22400001,11,4,1,11:37,Horford REBOUND (Off:0 Def:1),,
6,22400001,12,2,1,11:24,MISS White 28' 3PT Jump Shot,,
7,22400001,13,4,1,11:22,,Johnson REBOUND (Off:0 Def:1),
8,22400001,14,5,1,11:17,,Johnson Traveling Turnover (P1.T1),
9,22400001,15,5,1,10:57,Brown Bad Pass Turnover (P1.T1),Daniels STEAL (1 STL),


In [8]:
# Candidate natural key check for a raw play-by-play table.
pbp.duplicated(["GAME_ID", "EVENTNUM"]).sum()

2

In [9]:
duplicate_events = pbp[pbp.duplicated(["GAME_ID", "EVENTNUM"], keep=False)].sort_values(["GAME_ID", "EVENTNUM"])
duplicate_events[["GAME_ID", "EVENTNUM", "EVENTMSGTYPE", "PERIOD", "PCTIMESTRING", "HOMEDESCRIPTION", "VISITORDESCRIPTION", "SCORE", "SCOREMARGIN"]]

,GAME_ID,EVENTNUM,EVENTMSGTYPE,PERIOD,PCTIMESTRING,HOMEDESCRIPTION,VISITORDESCRIPTION,SCORE,SCOREMARGIN
224952,22400480,303,1,2,1:23,Turner 3' Layup (9 PTS) (Mathurin 3 AST),,54 - 54,TIE
224953,22400480,303,1,2,1:23,Turner 3' Layup (9 PTS) (Mathurin 3 AST),,54 - 54,TIE
224956,22400480,308,1,2,0:59,Mathurin 2' Running Finger Roll Layup (4 PTS) (Haliburton 4 AST),,54 - 56,2
224957,22400480,308,1,2,0:59,Mathurin 2' Running Finger Roll Layup (4 PTS) (Haliburton 4 AST),,54 - 56,2


## Date And Time Fields

There is no `GAME_DATE` column here. The file has wall-clock and period-clock strings.

In [10]:
time_cols = [col for col in pbp.columns if "DATE" in col.upper() or "TIME" in col.upper() or "CLOCK" in col.upper()]
time_cols

['WCTIMESTRING', 'PCTIMESTRING']

In [11]:
pbp[["GAME_ID", "PERIOD", "WCTIMESTRING", "PCTIMESTRING", "NEUTRALDESCRIPTION"]].head(20)

,GAME_ID,PERIOD,WCTIMESTRING,PCTIMESTRING,NEUTRALDESCRIPTION
0,22400001,1,7:11 PM,12:00,Start of 1st Period (7:11 PM EST)
1,22400001,1,7:11 PM,12:00,
2,22400001,1,7:11 PM,11:43,
3,22400001,1,7:11 PM,11:42,
4,22400001,1,7:11 PM,11:38,
5,22400001,1,7:11 PM,11:37,
6,22400001,1,7:11 PM,11:24,
7,22400001,1,7:11 PM,11:22,
8,22400001,1,7:11 PM,11:17,
9,22400001,1,7:12 PM,10:57,


Schema implication: shufinskiy alone is not enough to build exact `scheduled_tip_time_utc`. M1 game metadata needs `nba_api` schedule/box-score data or another source with game dates.

## Team Identity

Team identity is attached to player participant slots, not to one row-level `TEAM_ID` column.

In [12]:
team_cols = [col for col in pbp.columns if "TEAM" in col]
team_cols

['PLAYER1_TEAM_ID',
 'PLAYER1_TEAM_CITY',
 'PLAYER1_TEAM_NICKNAME',
 'PLAYER1_TEAM_ABBREVIATION',
 'PLAYER2_TEAM_ID',
 'PLAYER2_TEAM_CITY',
 'PLAYER2_TEAM_NICKNAME',
 'PLAYER2_TEAM_ABBREVIATION',
 'PLAYER3_TEAM_ID',
 'PLAYER3_TEAM_CITY',
 'PLAYER3_TEAM_NICKNAME',
 'PLAYER3_TEAM_ABBREVIATION']

In [13]:
team_frames = []
for slot in ["PLAYER1", "PLAYER2", "PLAYER3"]:
    cols = [f"{slot}_TEAM_ID", f"{slot}_TEAM_CITY", f"{slot}_TEAM_NICKNAME", f"{slot}_TEAM_ABBREVIATION"]
    frame = pbp.loc[pbp[f"{slot}_TEAM_ID"] != "", cols].drop_duplicates().copy()
    frame.columns = ["team_id", "city", "nickname", "abbreviation"]
    team_frames.append(frame)

teams = pd.concat(team_frames).drop_duplicates().sort_values("team_id").reset_index(drop=True)
teams.shape, teams.head(40)

((30, 4),
        team_id           city       nickname abbreviation
 0   1610612737        Atlanta          Hawks          ATL
 1   1610612738         Boston        Celtics          BOS
 2   1610612739      Cleveland      Cavaliers          CLE
 3   1610612740    New Orleans       Pelicans          NOP
 4   1610612741        Chicago          Bulls          CHI
 5   1610612742         Dallas      Mavericks          DAL
 6   1610612743         Denver        Nuggets          DEN
 7   1610612744   Golden State       Warriors          GSW
 8   1610612745        Houston        Rockets          HOU
 9   1610612746             LA       Clippers          LAC
 10  1610612747    Los Angeles         Lakers          LAL
 11  1610612748          Miami           Heat          MIA
 12  1610612749      Milwaukee          Bucks          MIL
 13  1610612750      Minnesota   Timberwolves          MIN
 14  1610612751       Brooklyn           Nets          BKN
 15  1610612752       New York         Knicks 

## Player Identity

Player identity also appears in participant slots. ID `0` means no real player in that slot.

In [14]:
player_cols = [col for col in pbp.columns if "PLAYER" in col]
player_cols

['PLAYER1_ID',
 'PLAYER1_NAME',
 'PLAYER1_TEAM_ID',
 'PLAYER1_TEAM_CITY',
 'PLAYER1_TEAM_NICKNAME',
 'PLAYER1_TEAM_ABBREVIATION',
 'PLAYER2_ID',
 'PLAYER2_NAME',
 'PLAYER2_TEAM_ID',
 'PLAYER2_TEAM_CITY',
 'PLAYER2_TEAM_NICKNAME',
 'PLAYER2_TEAM_ABBREVIATION',
 'PLAYER3_ID',
 'PLAYER3_NAME',
 'PLAYER3_TEAM_ID',
 'PLAYER3_TEAM_CITY',
 'PLAYER3_TEAM_NICKNAME',
 'PLAYER3_TEAM_ABBREVIATION']

In [15]:
player_frames = []
for slot in ["PLAYER1", "PLAYER2", "PLAYER3"]:
    cols = [f"{slot}_ID", f"{slot}_NAME", f"{slot}_TEAM_ID", f"{slot}_TEAM_ABBREVIATION"]
    frame = pbp.loc[pbp[f"{slot}_ID"] != "0", cols].drop_duplicates().copy()
    frame.columns = ["player_id", "player_name", "team_id", "team_abbreviation"]
    player_frames.append(frame)

players_seen = pd.concat(player_frames).drop_duplicates().sort_values("player_id").reset_index(drop=True)
players_seen.shape, players_seen.head(25)

((1440, 4),
    player_id         player_name     team_id team_abbreviation
 0         10                                                  
 1        101        Scott Skiles                              
 2     101108          Chris Paul  1610612759               SAS
 3        102                                                  
 4        103            Todd Day                              
 5        104     Benoit Benjamin                              
 6        105         Dan Majerle                              
 7        106                                                  
 8        108                                                  
 9        109        Robert Horry                              
 10        11          Pan Zhenqi                              
 11       110                                                  
 12       112  Dontonio Wingfield                              
 13       113                                                  
 14       114        Harold 

## Sparse Columns

Many columns are intentionally sparse because each event has only the fields relevant to that event.

In [16]:
blank_counts = (pbp == "").sum().sort_values(ascending=False)
blank_report = pd.DataFrame({
    "blank_count": blank_counts,
    "blank_pct": (blank_counts / len(pbp) * 100).round(2),
})
blank_report.head(20)

,blank_count,blank_pct
NEUTRALDESCRIPTION,561225,97.71
PLAYER3_TEAM_ABBREVIATION,560271,97.55
PLAYER3_TEAM_NICKNAME,560271,97.55
PLAYER3_TEAM_CITY,560271,97.55
PLAYER3_TEAM_ID,560271,97.55
PLAYER3_NAME,560271,97.55
SCOREMARGIN,418582,72.88
SCORE,418582,72.88
PLAYER2_TEAM_ABBREVIATION,381060,66.35
PLAYER2_TEAM_CITY,381060,66.35


## Event Types And Scores

`EVENTMSGTYPE` is the high-level event kind. `SCORE` and `SCOREMARGIN` appear only on score-bearing rows and period-end rows.

In [17]:
pbp["EVENTMSGTYPE"].value_counts().sort_index()

EVENTMSGTYPE
1     102568
10      2193
11        90
12      4984
13      4984
18      2902
2     116962
3      53312
4     128700
5      35174
6      46916
7       2114
8      59783
9      13678
Name: count, dtype: int64

In [18]:
score_rows = pbp.loc[pbp["SCORE"] != "", ["GAME_ID", "EVENTNUM", "EVENTMSGTYPE", "PERIOD", "PCTIMESTRING", "SCORE", "SCOREMARGIN", "HOMEDESCRIPTION", "VISITORDESCRIPTION", "NEUTRALDESCRIPTION"]]
score_rows.head(20)

,GAME_ID,EVENTNUM,EVENTMSGTYPE,PERIOD,PCTIMESTRING,SCORE,SCOREMARGIN,HOMEDESCRIPTION,VISITORDESCRIPTION,NEUTRALDESCRIPTION
12,22400001,21,1,1,10:50,3 - 0,-3,,Johnson 26' 3PT Jump Shot (3 PTS) (Wallace 1 AST),
13,22400001,23,1,1,10:35,3 - 3,TIE,Brown 27' 3PT Pullup Jump Shot (3 PTS) (White 1 AST),,
23,22400001,36,1,1,9:44,5 - 3,-2,,Capela 3' Putback Layup (2 PTS),
24,22400001,37,1,1,9:35,5 - 5,TIE,Brown 1' Driving Layup (5 PTS) (White 2 AST),,
25,22400001,39,1,1,9:23,7 - 5,-2,,Risacher 2' Cutting Layup Shot (2 PTS) (Wallace 2 AST),
27,22400001,43,3,1,9:11,7 - 6,-1,Brown Free Throw 1 of 2 (6 PTS),,
41,22400001,58,1,1,8:13,7 - 8,1,White 2' Cutting Layup Shot (2 PTS) (Tatum 1 AST),,
42,22400001,60,1,1,8:05,9 - 8,-1,,Capela 2' Alley Oop Dunk (4 PTS) (Johnson 1 AST),
56,22400001,77,1,1,7:02,11 - 8,-3,,Risacher 3' Driving Finger Roll Layup (4 PTS) (Capela 1 AST),
62,22400001,86,1,1,6:37,11 - 10,-1,Brown 1' Driving Dunk (8 PTS) (Holiday 1 AST),,


## One Game Example

Looking at one game makes the event shape obvious.

In [19]:
first_game_id = pbp["GAME_ID"].iloc[0]
first_game = pbp[pbp["GAME_ID"] == first_game_id]
first_game_id, first_game.shape

('22400001', (445, 34))

In [20]:
first_game[["EVENTNUM", "EVENTMSGTYPE", "PERIOD", "WCTIMESTRING", "PCTIMESTRING", "HOMEDESCRIPTION", "VISITORDESCRIPTION", "NEUTRALDESCRIPTION", "SCORE", "SCOREMARGIN"]].head(30)

,EVENTNUM,EVENTMSGTYPE,PERIOD,WCTIMESTRING,PCTIMESTRING,HOMEDESCRIPTION,VISITORDESCRIPTION,NEUTRALDESCRIPTION,SCORE,SCOREMARGIN
0,2,12,1,7:11 PM,12:00,,,Start of 1st Period (7:11 PM EST),,
1,4,10,1,7:11 PM,12:00,Jump Ball Horford vs. Capela: Tip to Wallace,,,,
2,7,2,1,7:11 PM,11:43,Tatum BLOCK (1 BLK),MISS Risacher 27' 3PT Jump Shot,,,
3,9,4,1,7:11 PM,11:42,,Risacher REBOUND (Off:1 Def:0),,,
4,10,2,1,7:11 PM,11:38,,MISS Johnson 14' Driving Floating Bank Jump Shot,,,
5,11,4,1,7:11 PM,11:37,Horford REBOUND (Off:0 Def:1),,,,
6,12,2,1,7:11 PM,11:24,MISS White 28' 3PT Jump Shot,,,,
7,13,4,1,7:11 PM,11:22,,Johnson REBOUND (Off:0 Def:1),,,
8,14,5,1,7:11 PM,11:17,,Johnson Traveling Turnover (P1.T1),,,
9,15,5,1,7:12 PM,10:57,Brown Bad Pass Turnover (P1.T1),Daniels STEAL (1 STL),,,


In [21]:
first_game[["EVENTNUM", "EVENTMSGTYPE", "PERIOD", "WCTIMESTRING", "PCTIMESTRING", "HOMEDESCRIPTION", "VISITORDESCRIPTION", "NEUTRALDESCRIPTION", "SCORE", "SCOREMARGIN"]].tail(10)

,EVENTNUM,EVENTMSGTYPE,PERIOD,WCTIMESTRING,PCTIMESTRING,HOMEDESCRIPTION,VISITORDESCRIPTION,NEUTRALDESCRIPTION,SCORE,SCOREMARGIN
435,646,1,4,9:17 PM,0:06,,Okongwu Tip Layup Shot (15 PTS),,117 - 116,-1
436,647,9,4,9:17 PM,0:06,CELTICS Timeout: Regular (Full 5 Short 0),,,,
437,648,6,4,9:19 PM,0:05,,Daniels P.FOUL (P3.T4) (R.Acosta),,,
438,650,9,4,9:19 PM,0:05,CELTICS Timeout: Regular (Full 6 Short 0),,,,
439,651,5,4,9:21 PM,0:04,Holiday Out of Bounds - Bad Pass Turnover Turnover (P2.T20),,,,
440,652,9,4,9:21 PM,0:04,,Hawks Timeout: Regular (Reg.7 Short 0),,,
441,653,5,4,9:23 PM,0:03,,Daniels Out of Bounds - Bad Pass Turnover Turnover (P6.T16),,,
442,655,2,4,9:24 PM,0:00,MISS Brown 13' Pullup Jump Shot,,,,
443,656,4,4,9:24 PM,0:00,CELTICS Rebound,,,,
444,657,13,4,9:24 PM,0:00,,,End of 4th Period (9:24 PM EST),117 - 116,-1


## M1 Schema Takeaway

Do not load this file into `player_game_logs`.

For M1:

- `player_game_logs` should come from `nba_api` box-score/player-log endpoints.
- shufinskiy should feed a raw `play_by_play_events` table if loaded now.
- The likely raw play-by-play natural key is `(GAME_ID, EVENTNUM)`.
- Every fact table still needs `event_time` and `ingested_at` so backtests can filter by point-in-time availability.